#AI Literacy Activity 2
by Maria Contreras

Given Task:

 In this exercise you will ask AI for a CNN twice on the same small image dataset: once with a bare prompt, once with the context a professional would give. You will run both, then use your own numbers to explain why the results differ.

## Set Up

In [6]:
!pip install -q importlib_resources

In [7]:
import tensorflow as tf
import tensorflow_datasets as tfds

def load_data():
  (train_ds, val_ds), info = tfds.load(
      "tf_flowers",
      split=["train[:80%]", "train[80%:]"],
      as_supervised=True,
      with_info=True,
  )

  IMG_SIZE = 160

  def prep(image, label):
      return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

  train_ds = train_ds.map(prep).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
  val_ds = val_ds.map(prep).batch(32).prefetch(tf.data.AUTOTUNE)

  class_names = info.features["label"].names
  return train_ds, val_ds, class_names, (IMG_SIZE, IMG_SIZE, 3)

## *PART 1: THE BARE PROMPT*

### Step 2. Assumption Table

Propmpt: Write Keras code for a CNN that classifies images.

|Detail	| What AI assumed Is that true for our dataset?|
|:------:|:--------------------------------------------:|
|Input image size |           128 * 128 *3              |
|Number of classes |           6 classes               |
|Label format (integer or one-hot) |int|
|loss function| sparse_categorical_crossentropy |
|Pixel scaling| Rescaling(1/255) mapping [0,255] to [0,1]|
|Which dataset, or how the data is loaded | data_dir/<class_name>/*.jpg with 80/20 split |
|Model architecture	|  ModelCheckpoit at best validation, early  stoping patience 8 on validation loss and reduce lr on plateau. defoult to 30 epochs|

### Step 3. Code Generated with minor ajustments

In [8]:
"""
CNN image classifier in Keras.

Usage:
  # Train on tf_flowers (train_ds / val_ds, downloads automatically):
  python cnn_classifier.py

  # Predict a single image with a trained model:
  python cnn_classifier.py --predict path/to/image.jpg
"""

import argparse
import json

import numpy as np
import keras
from keras import layers
import tensorflow as tf
import tensorflow_datasets as tfds

# ---------------------------------------------------------------- model
def conv_block(x, filters, dropout):
    """Two 3x3 convs + batch norm + pooling + dropout."""
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x)
    return layers.Dropout(dropout)(x)


def build_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Augmentation (active only during training)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)

    # Scale pixels from [0, 255] to [0, 1]
    x = layers.Rescaling(1.0 / 255)(x)

    x = conv_block(x, 32, 0.2)
    x = conv_block(x, 64, 0.3)
    x = conv_block(x, 128, 0.4)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="cnn_classifier")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

### Step 4. Train for 10 epochs record result

In [9]:
# ---------------------------------------------------------------- train
def train(args):
    train_ds, val_ds, class_names, input_shape = load_data()
    fit_kwargs = dict(x=train_ds, validation_data=val_ds)

    model = build_model(input_shape, len(class_names))
    model.summary()

    callbacks = [
        keras.callbacks.ModelCheckpoint(args.model_path, monitor="val_accuracy",
                                        save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=3, min_lr=1e-6),
    ]

    history = model.fit(epochs=10, callbacks=callbacks, **fit_kwargs)
    with open(args.model_path + ".history.json", "w") as f:
      json.dump(history.history, f)


    # Save class names alongside the model for prediction later
    with open(args.model_path + ".classes.json", "w") as f:
        json.dump({"class_names": class_names, "input_shape": input_shape}, f)

    print(f"Best model saved to {args.model_path}")

    return model


# ---------------------------------------------------------------- predict
def predict(args):
    model = keras.models.load_model(args.model_path)
    with open(args.model_path + ".classes.json") as f:
        meta = json.load(f)
    h, w = meta["input_shape"][:2]

    img = keras.utils.load_img(args.predict, target_size=(h, w))
    arr = np.expand_dims(keras.utils.img_to_array(img), 0)  # shape (1, h, w, 3)

    probs = model.predict(arr, verbose=0)[0]
    top = np.argsort(probs)[::-1][:3]
    for i in top:
        print(f"{meta['class_names'][i]:>15s}: {probs[i]:.2%}")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Keras CNN image classifier")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--model_path", default="cnn_classifier.keras")
    p.add_argument("--predict", help="Image path to classify with a trained model")
    # parse_known_args ignores extra args, e.g. the "-f kernel.json" Jupyter/Colab adds
    args, _ = p.parse_known_args()

    predict(args) if args.predict else train(args)

Model: "cnn_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 160, 160, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 160, 160, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 160, 160, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 160, 160, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 160, 160, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 80, 80, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 80, 80, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 40, 40, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 305,509 (1.17 MB)

 Trainable params: 304,613 (1.16 MB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 27s 172ms/step - accuracy: 0.4670 - loss: 1.2903 - val_accuracy: 0.2166 - val_loss: 1.9952 - learning_rate: 0.0010
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 154ms/step - accuracy: 0.5518 - loss: 1.1150 - val_accuracy: 0.2166 - val_loss: 2.1883 - learning_rate: 0.0010
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 156ms/step - accuracy: 0.5920 - loss: 1.0459 - val_accuracy: 0.2425 - val_loss: 2.2065 - learning_rate: 0.0010
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 157ms/step - accuracy: 0.6240 - loss: 0.9612 - val_accuracy: 0.3678 - val_loss: 1.6784 - learning_rate: 0.0010
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 158ms/step - accuracy: 0.6482 - loss: 0.9523 - val_accuracy: 0.6226 - val_loss: 0.9794 - learning_rate: 0.0010
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 158ms/step - accuracy: 0.6349 - loss: 0.9226 - val_accuracy: 0.6580 - val_loss: 0.9327 - learning_rate: 0.0010
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 15s 160ms/step - accuracy: 0.6550 - loss: 0.

## *PART 2: THE INFORMED PROMPT*

facts about this dataset (Needed for part 2):

* 3 670 flowers photos, 5 classes
* Images are $160 * 160 * 3$
* Pixels values range from 0 to 255
* Lables are integers 0 to 4 (not one-hot)

Step 1. Write your own prompt for the same task. This time describe your situation: the dataset facts listed in Setup, that you are on free Colab, and that your goal is the best validation accuracy you can get in 10 epochs or fewer.

Describe the problem, not the solution. Do not name a technique or architecture. Let AI decide what to recommend once it knows your situation.

A professional prompt for a task like this covers four things. Use this as a checklist, and write it in your own words:

1. Your data: how many images, how many classes, image size, pixel range, label format, and how it is already loaded
2. Your constraints: the compute and time you have
3. Your goal: what "good" means and how you will measure it
4. What you want back: runnable code, plus a short reason for the approach AI chose

Paste your exact prompt and AI's response into your notebook.

### Step 1. Prompt I created using the facts about the database.

I have 3670 flowers photos, 5 classes, the images are 160 by 160 by 3, pixel values range from 0 to 255, labels are integers 0 to 4 (no one-hot). this is how it's loaded:
```
def load_data():
  (train_ds, val_ds), info = tfds.load(
      "tf_flowers",
      split=["train[:80%]", "train[80%:]"],
      as_supervised=True,
      with_info=True,
  )
    IMG_SIZE = 160
    
  def prep(image, label):
      return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label
       
  train_ds = train_ds.map(prep).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
  val_ds = val_ds.map(prep).batch(32).prefetch(tf.data.AUTOTUNE)
```
I am using Google Colab free running on T4 GPU, Python 3. My goal is the best validation accuracy you can get in 10 epochs or fewer. I want runnable code that follows the stablish constraints as well a short reason for the approach you chose to use.

#### AI's Responce and Code

The script in flowers_efficientnetv2.py runs as one Colab cell. It keeps your load_data() exactly as written and only adds the return it was missing. It trains for exactly 10 epochs and prints the best validation accuracy at the end. On this split, this setup typically lands in the mid-to-high 90s, though the exact number varies from run to run.

Why this approach

Transfer learning instead of training from scratch. 2,936 training images and 10 epochs are far too few for a CNN trained from scratch, which usually plateaus around 70–80%. An ImageNet-pretrained backbone already recognizes petals, textures, and shapes, so most of the work is done before training starts.
EfficientNetV2-S as the backbone. It is one of the most accurate Keras backbones that still fine-tunes comfortably at 160×160 on a T4’s 16 GB. Its built-in preprocessing takes your raw 0–255 pixels directly, so your data pipeline doesn’t change.
Training in two phases (3 + 7 epochs).
First, only the new classifier head trains, at a high learning rate. This keeps random gradients from damaging the pretrained weights.
Then the whole network is unfrozen and fine-tuned with a small learning rate (3e-5) that decays toward zero on a cosine schedule. This is where most of the accuracy gain past ~90% comes from.
BatchNorm layers stay in inference mode throughout (training=False), which is the standard way to keep fine-tuning stable on small datasets.
Light augmentation and dropout. Random flips, rotations, and zooms plus a dropout layer reduce overfitting on a small dataset without slowing the model’s convergence.
Mixed precision. It uses the T4’s tensor cores, so each epoch takes roughly half the time.
A checkpoint on val_accuracy. The best epoch’s weights are saved, so you get the peak result even if the last epoch dips slightly.

The labels stay as integers 0–4 throughout, using SparseCategoricalCrossentropy, so no one-hot encoding is needed.

In [11]:
# tf_flowers: best val accuracy in <= 10 epochs (Colab free, T4)
# Transfer learning: EfficientNetV2-S (ImageNet) -> 3 epochs head-only, 7 epochs full fine-tune.

import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers

keras.utils.set_random_seed(42)
keras.mixed_precision.set_global_policy("mixed_float16")  # ~2x faster on T4 tensor cores


# ---- Your loader (unchanged, plus a return statement) ----
def load_data():
    (train_ds, val_ds), info = tfds.load(
        "tf_flowers",
        split=["train[:80%]", "train[80%:]"],
        as_supervised=True,
        with_info=True,
    )

    IMG_SIZE = 160

    def prep(image, label):
        return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

    train_ds = train_ds.map(prep).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.map(prep).batch(32).prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds, info


train_ds, val_ds, info = load_data()
NUM_CLASSES = info.features["label"].num_classes  # 5

HEAD_EPOCHS = 3
TOTAL_EPOCHS = 10  # hard cap

# ---- Model ----
# include_preprocessing=True: the backbone takes raw 0-255 pixels, so no manual rescaling.
base = keras.applications.EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(160, 160, 3),
    include_preprocessing=True,
)
base.trainable = False

augment = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.15),
    ],
    name="augment",
)

inputs = keras.Input(shape=(160, 160, 3))
x = augment(inputs)                      # active only during training
x = base(x, training=False)              # keeps BatchNorm in inference mode, even after unfreezing
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)  # float32 output for stability
model = keras.Model(inputs, outputs)

loss = keras.losses.SparseCategoricalCrossentropy()  # integer labels 0-4

ckpt = keras.callbacks.ModelCheckpoint(
    "best.weights.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)

# ---- Phase 1: train the new head only (backbone frozen) ----
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=[ckpt])

# ---- Phase 2: unfreeze everything, fine-tune with a small, decaying LR ----
base.trainable = True
steps_per_epoch = int(train_ds.cardinality())
if steps_per_epoch <= 0:  # fallback if cardinality is unknown
    steps_per_epoch = 92  # ceil(2936 / 32)

lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-5,
    decay_steps=steps_per_epoch * (TOTAL_EPOCHS - HEAD_EPOCHS),
)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
    loss=loss,
    metrics=["accuracy"],
)
model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=HEAD_EPOCHS,
    epochs=TOTAL_EPOCHS,  # epochs 4-10 -> 10 total
    callbacks=[ckpt],
)

# ---- Report best validation accuracy ----
model.load_weights("best.weights.h5")
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Best validation accuracy: {val_acc:.4f}")

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.5362 - loss: 1.1791
Epoch 1: val_accuracy improved from None to 0.83379, saving model to best.weights.h5

Epoch 1: finished saving model to best.weights.h5
92/92 ━━━━━━━━━━━━━━━━━━━━ 44s 188ms/step - accuracy: 0.6740 - loss: 0.9082 - val_accuracy: 0.8338 - val_loss: 0.5234
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8103 - loss: 0.5831
Epoch 2: val_accuracy improved from 0.83379 to 0.86240, saving model to best.weights.h5

Epoch 2: finished saving model to best.weights.h5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 110ms/step - accuracy: 0.8110 - loss: 0.5532 - val_accuracy: 0.8624 - val_loss: 0.4147
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.8449 - loss: 0.4687
Epoch 3: val_accuracy improved from 0.86240 to 0.87330, saving model to best.weights.h5

Epoch 3: finished saving model to best.weights.h5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 102ms/step - acc

#### Step 2.

Run the code on the same train_ds and val_ds. Record your results in the Part 3 table. Run model.summary() and keep the output.

In [12]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augment (Sequential)            │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 5, 5, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,705,556 (231.57 MB)

 Trainable params: 20,183,893 (77.00 MB)

 Non-trainable params: 153,872 (601.06 KB)

 Optimizer params: 40,367,791 (153.99 MB)

## *PART 3: COMPARE AND EXPLAIN*

Results Table comparing model 1 and 2.
 Total params: 60,705,556 (231.57 MB)
 Trainable params: 20,183,893 (77.00 MB)
 Non-trainable params: 153,872 (601.06 KB)
 Optimizer params: 40,367,791 (153.99 MB)

|                            | Part 1 | Part 2 |
|:--------------------------:|:-------------------------:|:-------------------------------------------:|
| Final training accuracy    | 0.6846 (68.46%)           | 0.8900 (89.00%)                             |
| Final validation accuracy  | 0.6689 (66.89%)           | 0.8924 (89.24%) — best: 0.8978 at epoch 9   |
| Train minus validation gap | +0.0157 (+1.57 pts)       | −0.0024 (−0.24 pts)                         |
| Total parameters           | 305,509 (1.17 MB)         | 20,337,765 (77.58 MB)                       |
| Trainable parameters       | 304,613 (1.16 MB)         | 20,183,893 (77.00 MB)                       |
| Approx. time per epoch     | ~15 s (~158 ms/step)      | ~10 s frozen (epochs 2–3), ~38 s fine-tuning (epochs 5–10) |

#### Explain:
2. **Why the difference?** Which model did better on validation, and by how much? Using this week's concepts, explain why. Refer to your gap and parameter rows, not to general statements.

As expected The model in part 2 did way better than the one from part 1 by 21.46%. Given my gap row for part 1; +`1.57 pts, it indicates high training acuracy but failure at validation giving underfiting, copared the the -0.24 pts form part 2, which means high validation to low training ration, which would represent overfiting if it where a higher percentage. But the gap in both is still tiny so for part 2 it's neither. Part 1 also has 304 613 trainable parameters that is to high a number to train on 10 epochs over 2936 images.

3. **Trainable vs. total.** Look at the trainable and total parameter counts for each model. If they differ for one of your models, what explains it, and what does that tell you about which parts of the model learned from your flower images?

THe differ slighly for both of my models, the first by 896 and the second by 153, 672. This is due to the batch normalization layers wich store four numbers per channel. The gamma (scale) and beta (shift) which learn by gradient descent, so they are counted as trainable. Then there are the moving mean and moving variance, which are averages of the data's statistics, backpropagation never updates them so keras counts them as non-tarinable. At least for part 2; epochs 4 two 6 is when the model most learned. The first model had a lot less of the untrainable, so it consentrated a lot less on statistics.

4. **What changed AI's answer?** Which single piece of context in your Part 2 prompt do you think changed AI's response the most? What makes you think so?

Some things I noticed affected the way that AI the most was stating how I wanted the model to run on 10 or less epochs. I belive this along with the specifics of the data where the bigest impact since the number of epochs actually affect a lot the way models train.

5. **Honest check.** If Part 2 did not beat Part 1, if AI gave you nearly the same approach both times, or if something in your results does not make sense to you, say so here and describe what you think happened.

For me Part 2 beat Part 1 first try. Arguably, my Part 1 is horrible so even a fish could get a better score.